# Airbnb Paris – Text-Embeddings für die Ausreißererkennung
- Vergleich der Baseline-Detektoren (iForest, LODA, ECOD, AutoEncoder) über drei Repräsentationen
- Gleiche numerische Basis (`cleaned`); Text einmal weggelassen, einmal per **fastText**, einmal per **Sentence-Transformer** (`semantic_pca30`)
- Beste Hyperparameter aus der README (kein GridSearch); kein MLflow-Tracking

In [3]:
import time
import numpy as np
import pandas as pd
import fasttext
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc, classification_report
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

## Daten & gemeinsamer Split
- Outlier = `is_top_rating == 0`; 3 Preprocessing-Varianten über `row_id` alignt, 70/30 stratifiziert (seed 42) → identisches Test-Set

In [4]:
ds = "airbnb_paris"
load = lambda name: pd.read_csv(f"../data/preprocessed/{name}_{ds}.csv").set_index("row_id")

cleaned = load("cleaned")
cleaned_text = load("cleaned_text")
semantic_pca30 = load("semantic_pca30")

common = cleaned.index.intersection(semantic_pca30.index).sort_values()
y = (1 - cleaned.loc[common, "is_top_rating"]).values
print("rows", len(common), "outlier rate", round(y.mean(), 4))

tr_id, te_id = train_test_split(common, test_size=0.3, stratify=y, random_state=42)
y_train = (1 - cleaned.loc[tr_id, "is_top_rating"]).values
y_test = (1 - cleaned.loc[te_id, "is_top_rating"]).values

rows 18350 outlier rate 0.0393


## FastText-Embeddings
- 4 Freitextspalten pro Zeile zu einem Dokument verbunden
- Unsupervised skipgram (dim=100) auf dem Korpus, dann Satzvektor je Zeile, StandardScaler

In [5]:
text_cols = ["name", "description", "neighborhood_overview", "host_about"]
docs = cleaned_text.loc[common, text_cols].fillna("").apply(lambda r: " ".join(r), axis=1).str.replace(r"[\r\n]+", " ", regex=True)

with open("/tmp/ft_corpus.txt", "w") as f:
    f.write("\n".join(docs.tolist()))

ft = fasttext.train_unsupervised("/tmp/ft_corpus.txt", model="skipgram", dim=100)
emb = np.vstack([ft.get_sentence_vector(t) for t in docs])
print("fasttext embeddings", emb.shape)

Read 2M words
Number of words:  17639
Number of labels: 0
24.9% words/sec/thread:   81867 lr:  0.037545 avg.loss:  2.090149 ETA:   0h 0m 8s 29.6% words/sec/thread:   82948 lr:  0.035213 avg.loss:  2.070969 ETA:   0h 0m 8sProgress:  38.7% words/sec/thread:   83878 lr:  0.030655 avg.loss:  2.042405 ETA:   0h 0m 7s 54.9% words/sec/thread:   85913 lr:  0.022541 avg.loss:  1.994580 ETA:   0h 0m 5s


fasttext embeddings (18350, 100)


In [6]:
emb = StandardScaler().fit_transform(emb)
ft_df = pd.DataFrame(emb, index=common, columns=[f"ft_{i}" for i in range(emb.shape[1])])

## Drei Repräsentationen
- Numerische Basis (`cleaned`) für alle identisch; Unterschied nur in der Text-Repräsentation

In [7]:
num = cleaned.drop(columns=["is_top_rating"]).loc[common]
pca = semantic_pca30[[c for c in semantic_pca30.columns if c.startswith("pca_")]].loc[common]

reps = {
    "no_text": num,
    "fasttext": num.join(ft_df),
    "semantic_pca30": num.join(pca),
}
for k, v in reps.items():
    print(f"{k:16s} features={v.shape[1]}")

no_text          features=40
fasttext         features=140
semantic_pca30   features=70


## Detektoren & Evaluation
- Beste Hyperparameter aus der README (Airbnb Paris), kein GridSearch
- Pro Modell: AP, AUPRC, AUC-ROC, Classification Report

In [8]:
CONT = round(y.mean(), 4)
detectors = {
    "iforest": (IForest, {"n_estimators": 100, "max_features": 1.0, "contamination": CONT, "random_state": 42}),
    "loda": (LODA, {"n_bins": 10, "n_random_cuts": 100, "contamination": CONT}),
    "ecod": (ECOD, {"contamination": CONT}),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [64, 32], "epoch_num": 50, "contamination": CONT, "random_state": 42, "device": "cuda"}),
}

results = []
for rep_name, rep in reps.items():
    Xtr, Xte = rep.loc[tr_id].values, rep.loc[te_id].values
    for det_name, (Model, params) in detectors.items():
        t0 = time.perf_counter()
        model = Model(**params)
        model.fit(Xtr)
        scores = model.decision_function(Xte)
        pred = model.predict(Xte)
        runtime = time.perf_counter() - t0
        ap = average_precision_score(y_test, scores)
        prec, rec, _ = precision_recall_curve(y_test, scores)
        auprc = auc(rec, prec)
        auroc = roc_auc_score(y_test, scores)
        results.append({"pipeline": rep_name, "detector": det_name, "AP": ap, "AUPRC": auprc, "AUC_ROC": auroc})
        print(f"\n=== {rep_name} | {det_name} (feat={rep.shape[1]}, t={runtime:.1f}s) ===")
        print(f"AP={ap:.4f}  AUPRC={auprc:.4f}  AUC-ROC={auroc:.4f}")
        print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4))


=== no_text | iforest (feat=40, t=0.4s) ===
AP=0.0886  AUPRC=0.0867  AUC-ROC=0.6962
              precision    recall  f1-score   support

      inlier     0.9651    0.9614    0.9633      5289
     outlier     0.1356    0.1481    0.1416       216

    accuracy                         0.9295      5505
   macro avg     0.5503    0.5548    0.5524      5505
weighted avg     0.9325    0.9295    0.9310      5505


=== no_text | loda (feat=40, t=0.1s) ===
AP=0.1126  AUPRC=0.1104  AUC-ROC=0.7246
              precision    recall  f1-score   support

      inlier     0.9660    0.9656    0.9658      5289
     outlier     0.1651    0.1667    0.1659       216

    accuracy                         0.9342      5505
   macro avg     0.5655    0.5661    0.5658      5505
weighted avg     0.9345    0.9342    0.9344      5505


=== no_text | ecod (feat=40, t=0.7s) ===
AP=0.1085  AUPRC=0.1058  AUC-ROC=0.7240
              precision    recall  f1-score   support

      inlier     0.9647    0.9662    0.965

Training: 100%|██████████| 50/50 [01:03<00:00,  1.27s/it]



=== no_text | autoencoder (feat=40, t=69.9s) ===
AP=0.0735  AUPRC=0.0711  AUC-ROC=0.6114
              precision    recall  f1-score   support

      inlier     0.9628    0.9686    0.9657      5289
     outlier     0.0978    0.0833    0.0900       216

    accuracy                         0.9339      5505
   macro avg     0.5303    0.5260    0.5278      5505
weighted avg     0.9289    0.9339    0.9313      5505


=== fasttext | iforest (feat=140, t=0.4s) ===
AP=0.0781  AUPRC=0.0760  AUC-ROC=0.6566
              precision    recall  f1-score   support

      inlier     0.9642    0.9612    0.9627      5289
     outlier     0.1164    0.1250    0.1205       216

    accuracy                         0.9284      5505
   macro avg     0.5403    0.5431    0.5416      5505
weighted avg     0.9309    0.9284    0.9297      5505


=== fasttext | loda (feat=140, t=0.2s) ===
AP=0.0696  AUPRC=0.0684  AUC-ROC=0.5882
              precision    recall  f1-score   support

      inlier     0.9639    0.9

Training: 100%|██████████| 50/50 [01:04<00:00,  1.29s/it]



=== fasttext | autoencoder (feat=140, t=65.2s) ===
AP=0.0698  AUPRC=0.0681  AUC-ROC=0.6043
              precision    recall  f1-score   support

      inlier     0.9637    0.9628    0.9632      5289
     outlier     0.1086    0.1111    0.1098       216

    accuracy                         0.9293      5505
   macro avg     0.5361    0.5369    0.5365      5505
weighted avg     0.9301    0.9293    0.9297      5505


=== semantic_pca30 | iforest (feat=70, t=0.4s) ===
AP=0.0801  AUPRC=0.0783  AUC-ROC=0.6684
              precision    recall  f1-score   support

      inlier     0.9637    0.9633    0.9635      5289
     outlier     0.1101    0.1111    0.1106       216

    accuracy                         0.9299      5505
   macro avg     0.5369    0.5372    0.5371      5505
weighted avg     0.9302    0.9299    0.9300      5505


=== semantic_pca30 | loda (feat=70, t=0.1s) ===
AP=0.0427  AUPRC=0.0417  AUC-ROC=0.4778
              precision    recall  f1-score   support

      inlier     0

Training: 100%|██████████| 50/50 [01:06<00:00,  1.32s/it]



=== semantic_pca30 | autoencoder (feat=70, t=67.0s) ===
AP=0.0523  AUPRC=0.0507  AUC-ROC=0.5268
              precision    recall  f1-score   support

      inlier     0.9617    0.9550    0.9584      5289
     outlier     0.0593    0.0694    0.0640       216

    accuracy                         0.9203      5505
   macro avg     0.5105    0.5122    0.5112      5505
weighted avg     0.9263    0.9203    0.9233      5505



## Ergebnisvergleich
- Mittelwert je Pipeline über alle Detektoren → beste Text-Repräsentation

In [9]:
res = pd.DataFrame(results)
print(res.round(4).to_string(index=False))

summary = res.groupby("pipeline")[["AP", "AUPRC", "AUC_ROC"]].mean().sort_values("AUPRC", ascending=False)
print("\nMittelwert über alle Detektoren:")
print(summary.round(4).to_string())
print("\nBeste Pipeline (nach AUPRC):", summary.index[0])

      pipeline    detector     AP  AUPRC  AUC_ROC
       no_text     iforest 0.0886 0.0867   0.6962
       no_text        loda 0.1126 0.1104   0.7246
       no_text        ecod 0.1085 0.1058   0.7240
       no_text autoencoder 0.0735 0.0711   0.6114
      fasttext     iforest 0.0781 0.0760   0.6566
      fasttext        loda 0.0696 0.0684   0.5882
      fasttext        ecod 0.0759 0.0739   0.6701
      fasttext autoencoder 0.0698 0.0681   0.6043
semantic_pca30     iforest 0.0801 0.0783   0.6684
semantic_pca30        loda 0.0427 0.0417   0.4778
semantic_pca30        ecod 0.0882 0.0857   0.6847
semantic_pca30 autoencoder 0.0523 0.0507   0.5268

Mittelwert über alle Detektoren:
                    AP   AUPRC  AUC_ROC
pipeline                               
no_text         0.0958  0.0935   0.6890
fasttext        0.0733  0.0716   0.6298
semantic_pca30  0.0658  0.0641   0.5894

Beste Pipeline (nach AUPRC): no_text
